# 💬 멀티턴 대화 구현 + 품질 불일치 관찰

---

## 🎯 오늘의 상황

> 파라미터 실험까지 마친 법률 AI 스타트업 개발자.  
> 팀장이 말합니다: "GPT가 이전 대화를 기억하질 못해요.  
> '아까 말한 내용 요약해줘' 하면 '무슨 내용인지 모르겠다'고 합니다."
>
> 문제의 원인은 **싱글턴 구조**입니다.  
> 오늘은 이전 대화를 기억하는 **멀티턴 챗봇**을 직접 만들어봅니다.

---

## 📋 실습 순서

```
STEP 1. 싱글턴 vs 멀티턴 차이 확인
STEP 2. chat() 함수 — 멀티턴의 핵심
STEP 3. safe_api_call — 실패해도 안전하게
STEP 4. 품질 불일치 관찰 실험
STEP 5. 히스토리 관리 주의사항
```

---

## 💡 핵심 개념

> **LLM은 메모리가 없다!**  
> GPT가 맥락을 기억하는 것처럼 보이는 건  
> 이전 대화를 **매번 통째로 다시 보내기** 때문입니다.

| 구조 | messages 구성 | 맥락 기억 | 토큰 비용 |
|------|--------------|----------|----------|
| **싱글턴** | 매번 새로 생성 | ❌ 없음 | 일정 |
| **멀티턴** | 이전 내용 누적 | ✅ 있음 | 턴마다 증가 |

---
## 🔧 사전 준비

In [9]:
# 이전 실습에서 사용한 클라이언트 + 비용 추적기 그대로 사용
from google.colab import userdata
import openai, time, json

client = openai.OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

class TokenTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.call_count = 0

    def add(self, response):
        self.total_input  += response.usage.prompt_tokens
        self.total_output += response.usage.completion_tokens
        self.call_count   += 1

    def report(self):
        cost = (self.total_input * 0.15 + self.total_output * 0.60) / 1_000_000
        print(f"총 {self.call_count}회 호출 | 입력 {self.total_input} + 출력 {self.total_output} 토큰")
        print(f"누적 비용: ${cost:.4f} (약 {cost * 1400:.0f}원)")

tracker = TokenTracker()
print("✅ 준비 완료!")

✅ 준비 완료!


---
## STEP 1. 싱글턴 vs 멀티턴 차이 확인

먼저 **싱글턴이 왜 문제인지** 직접 확인합니다.  
"방금 말한 내용 요약해줘" 라고 했을 때 어떻게 다른지 보세요.

In [10]:
# ─── 싱글턴 방식 ─── 매번 새로운 messages
system = {"role": "system", "content": "당신은 법률 전문 AI입니다."}

# 첫 번째 질문
r1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[system, {"role": "user", "content": "계약서에서 가장 중요한 조항은?"}],
    temperature=0.3, max_tokens=200,
)
print("[싱글턴 1턴] 계약서 조항 질문")
print(r1.choices[0].message.content[:150])

# 두 번째 질문 — 맥락을 이어가려 했지만...
r2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[system, {"role": "user", "content": "방금 말한 내용을 요약해 주세요."}],  # 새 messages
    temperature=0.3, max_tokens=200,
)
print("\n[싱글턴 2턴] 요약 요청 → 이전 대화를 모릅니다!")
print(r2.choices[0].message.content[:150])

tracker.add(r1)
tracker.add(r2)

[싱글턴 1턴] 계약서 조항 질문
계약서에서 가장 중요한 조항은 계약의 목적과 내용을 명확히 규정하는 "주요 조항"입니다. 일반적으로 다음과 같은 조항들이 중요하게 여겨집니다:

1. **계약의 목적**: 계약의 기본적인 목적과 취지를 명확히 설명합니다.
2. **당사자의 의무**: 각 당사자가 수행해

[싱글턴 2턴] 요약 요청 → 이전 대화를 모릅니다!
저는 법률 전문 AI로, 2023년 10월까지의 데이터에 기반하여 정보를 제공합니다. 법률 관련 질문에 대해 도움을 드릴 수 있습니다.


---
## STEP 2. chat() 함수 — 멀티턴의 핵심

멀티턴의 비결은 단순합니다.  
**대화할 때마다 이전 내용을 리스트에 쌓아서 함께 보내는 것**입니다.

```
초기:  [system]
1턴:   [system, user_1]          → API 호출 → [system, user_1, assistant_1]
2턴:   [system, user_1, assistant_1, user_2] → API 호출 → ... assistant_2 추가
```

In [11]:
# ① 대화 히스토리 초기화 — system 프롬프트로 시작
conversation_history = [
    {"role": "system", "content": "당신은 법률 전문 AI입니다. 쉽게 설명하세요."}
]

# ② chat 함수 — 4단계로 동작
def chat(user_input):
    # 1단계: user 메시지를 히스토리에 추가
    conversation_history.append({"role": "user", "content": user_input})

    # 2단계: 히스토리 전체를 API에 전달
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=conversation_history,   # 전체 대화 내용을 보냄
        temperature=0.3,
        max_tokens=300,
    )

    # 3단계: 응답을 히스토리에 추가
    reply = response.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": reply})

    # 4단계: 비용 추적 + 응답 반환
    tracker.add(response)
    return reply

print("✅ chat() 함수 준비 완료")

✅ chat() 함수 준비 완료


In [12]:
# ③ 3턴 대화 실행 — 세 번째 질문에서 맥락이 연결되는지 확인!
print("=" * 50)
print("[1턴]", chat("계약서에서 가장 중요한 조항이 무엇인가요?"))

print("\n" + "=" * 50)
print("[2턴]", chat("그렇다면 해지 조항은 어떻게 작성해야 하나요?"))

print("\n" + "=" * 50)
print("[3턴]", chat("방금 말한 내용을 요약해 주세요."))  # ← 1·2턴 내용을 참조해야 함!

[1턴] 계약서에서 가장 중요한 조항은 계약의 목적과 조건을 명확히 하는 "주요 조항"입니다. 이 조항은 계약의 핵심 내용인 거래의 내용, 가격, 납기일, 품질 기준 등을 포함합니다. 

또한, 다음과 같은 조항들도 중요합니다:

1. **지급 조건**: 대금 지급 방식과 기한.
2. **위험 부담**: 손해가 발생했을 때의 책임 분담.
3. **해지 조항**: 계약을 종료할 수 있는 조건.
4. **분쟁 해결**: 분쟁 발생 시 해결 방법 및 관할 법원.

이러한 조항들은 계약의 이행과 관련된 중요한 사항들을 규정하므로, 계약 체결 시 반드시 주의 깊게 검토해야 합니다.

[2턴] 해지 조항을 작성할 때는 다음과 같은 요소들을 포함하여 명확하고 구체적으로 작성하는 것이 중요합니다:

1. **해지 사유**: 계약을 해지할 수 있는 구체적인 사유를 명시합니다. 예를 들어, 계약 위반, 지급 지연, 파산 등의 사유를 포함할 수 있습니다.

2. **해지 통지 방법**: 계약 해지를 원할 경우 상대방에게 어떤 방식으로 통지해야 하는지를 명시합니다. 예를 들어, 서면 통지, 이메일 통지 등.

3. **해지 통지 기간**: 해지를 통지한 후 실제 해지가 이루어지기까지의 기간을 정합니다. 예를 들어, "해지 통지 후 30일 이내에 계약이 종료된다"와 같이.

4. **해지 후의 의무**: 계약 해지 후 양 당사자가 지켜야 할 의무를 명시합니다. 예를 들어, 기밀 유지 의무, 반환해야 할 자산 등에 대한 내용을 포함할 수 있습니다.

5. **해지의 효과**: 계약 해지 후 발생하는 결과에 대해 설명합니다. 예를 들어, 지급해야 할 잔여 금액, 손해배상 책임 등이 포함될 수 있습니다.

이러한 요소들을 포함하여 해지 조항을 작성하면, 계약 해지 시 발생할 수 있는 분쟁을 예방

[3턴] 해지 조항을 작성할 때 포함해야 할 주요 요소는 다음과 같습니다:

1. **해지 사유**: 계약 해지 사유를 구체적으로 명시.
2. **해지 통지 방법**: 해지를 통지할 방식(서면,

In [13]:
# ④ 히스토리 구조 확인 — 어떻게 쌓였는지 눈으로 보기
print(f"📊 히스토리 길이: {len(conversation_history)}개")
print(f"   → system(1) + user·assistant × 3턴 = 7개")

# role 순서 출력
roles = [item["role"] for item in conversation_history]
print(f"\n📋 role 순서: {roles}")

📊 히스토리 길이: 7개
   → system(1) + user·assistant × 3턴 = 7개

📋 role 순서: ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant']


---
## STEP 3. safe_api_call — 실패해도 안전하게

API 호출은 언제든 실패할 수 있습니다. (네트워크 오류, 서버 과부하 등)  
실패했을 때 **히스토리가 망가지지 않도록** 안전 장치를 만들어봅니다.

**지수 백오프(Exponential Backoff):** 실패할수록 더 오래 기다렸다가 재시도
```
1차 실패 → 1초 대기 → 재시도
2차 실패 → 2초 대기 → 재시도  
3차 실패 → 4초 대기 → 포기
```

In [14]:
# ① 안전한 API 호출 함수
def safe_api_call(messages, temperature=0.3, max_tokens=300):
    """실패 시 최대 3회까지 재시도, 그래도 안 되면 None 반환"""

    for attempt in range(3):   # 최대 3번 시도
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            tracker.add(response)                       # 성공 시 비용 추적
            return response.choices[0].message.content  # 성공 시 응답 반환

        except Exception as e:
            wait = 2 ** attempt   # 1초 → 2초 → 4초
            print(f"⚠️ [{attempt+1}/3] 오류: {e} → {wait}초 후 재시도")
            time.sleep(wait)

    print("❌ 최대 재시도 횟수 초과")
    return None   # 3번 모두 실패하면 None 반환

print("✅ safe_api_call() 준비 완료")

✅ safe_api_call() 준비 완료


In [15]:
# ② safe 버전 chat 함수 — 실패 시 히스토리 복구
def chat_safe(user_input):
    # user 메시지를 히스토리에 추가
    conversation_history.append({"role": "user", "content": user_input})

    # 안전하게 호출
    reply = safe_api_call(conversation_history)

    if reply is None:
        conversation_history.pop()   # 실패 시 방금 추가한 user 메시지 제거 (히스토리 보호)
        return "⚠️ 응답을 받지 못했습니다. 다시 시도해 주세요."

    conversation_history.append({"role": "assistant", "content": reply})
    return reply

print("✅ chat_safe() 준비 완료")

✅ chat_safe() 준비 완료


---
## STEP 4. 품질 불일치 관찰 실험

**같은 질문, 같은 설정으로 3번 호출하면 항상 같은 품질의 답이 나올까요?**  

답은 **아닙니다**. 이 현상을 직접 관찰하고 기록합니다.  
이 기록은 **Day 2 프롬프트 기법 실험의 입력 데이터**로 사용됩니다.

> ⚠️ **이 실험 결과는 반드시 저장하세요! Day 2에서 사용합니다.**

In [16]:
# 같은 질문을 temperature=0.3으로 3번 호출
test_question = "임차인이 보증금을 돌려받지 못할 때 법적으로 어떤 조치를 취할 수 있나요?"

quality_samples = []   # 3개 응답을 저장할 리스트

for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "당신은 법률 전문 AI입니다."},
            {"role": "user",   "content": test_question}
        ],
        temperature=0.3,   # 낮은 값이지만 완전히 같지 않을 수 있음
        max_tokens=300,
    )
    answer = response.choices[0].message.content
    quality_samples.append(answer)
    tracker.add(response)

    print(f"{'─'*50}")
    print(f"[{i+1}회차]")
    print(answer[:200])

──────────────────────────────────────────────────
[1회차]
임차인이 보증금을 돌려받지 못할 경우, 법적으로 취할 수 있는 조치는 다음과 같습니다:

1. **임대인과의 협상**: 우선 임대인과 직접 대화를 시도하여 보증금 반환에 대한 문제를 해결할 수 있습니다. 이때, 관련 서류(계약서, 입증 자료 등)를 준비하여 자신의 주장을 뒷받침하는 것이 중요합니다.

2. **내용증명 발송**: 임대인이 협조하지 않을 경우
──────────────────────────────────────────────────
[2회차]
임차인이 보증금을 돌려받지 못할 경우, 법적으로 취할 수 있는 조치는 다음과 같습니다:

1. **임대인과의 협의**: 우선 임대인과 직접 대화를 시도하여 보증금 반환에 대한 문제를 해결할 수 있는지 논의합니다. 이때, 서면으로 요청하는 것이 좋습니다.

2. **내용증명 발송**: 임대인과의 협의가 실패할 경우, 내용증명을 통해 보증금 반환을 요구하는 서
──────────────────────────────────────────────────
[3회차]
임차인이 보증금을 돌려받지 못할 경우, 법적으로 취할 수 있는 조치는 다음과 같습니다:

1. **임대인과의 협상**: 먼저 임대인과 직접 대화를 시도하여 보증금 반환에 대한 문제를 해결할 수 있습니다. 이때, 임대차 계약서와 관련 서류를 준비하여 자신의 주장을 뒷받침할 수 있는 자료를 제시하는 것이 좋습니다.

2. **내용증명 발송**: 임대인과의 협상


In [17]:
# 3개 응답의 차이를 자동으로 분석
print("📋 품질 불일치 분석")
print("=" * 50)

for i, ans in enumerate(quality_samples):
    has_law       = "조" in ans or "법" in ans                          # 법조문 인용 여부
    has_structure = any(c in ans for c in ["1.", "①", "첫째", "- "])   # 구조화 여부

    print(f"\n[{i+1}회차] 길이: {len(ans)}자")
    print(f"  법조문 인용: {'✅' if has_law else '❌'}  구조화: {'✅' if has_structure else '❌'}")
    print(f"  시작: {ans[:60]}...")

# 동일 여부 확인
if quality_samples[0] == quality_samples[1] == quality_samples[2]:
    print("\n→ 3회 모두 동일")
else:
    print("\n→ ⚠️ 답변이 다릅니다! 이것이 '품질 불일치' 현상입니다.")

📋 품질 불일치 분석

[1회차] 길이: 548자
  법조문 인용: ✅  구조화: ✅
  시작: 임차인이 보증금을 돌려받지 못할 경우, 법적으로 취할 수 있는 조치는 다음과 같습니다:

1. **임대인과의...

[2회차] 길이: 518자
  법조문 인용: ✅  구조화: ✅
  시작: 임차인이 보증금을 돌려받지 못할 경우, 법적으로 취할 수 있는 조치는 다음과 같습니다:

1. **임대인과의...

[3회차] 길이: 545자
  법조문 인용: ✅  구조화: ✅
  시작: 임차인이 보증금을 돌려받지 못할 경우, 법적으로 취할 수 있는 조치는 다음과 같습니다:

1. **임대인과의...

→ ⚠️ 답변이 다릅니다! 이것이 '품질 불일치' 현상입니다.


In [ ]:
# 관찰 결과를 JSON으로 저장 → Day 2에서 사용
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/llm_course/day1", exist_ok=True)

quality_note = {
    "question": test_question,
    "samples": quality_samples,
    "observation": "여기에 관찰 내용을 직접 작성하세요"   # ← 직접 수정!
}

with open("/content/drive/MyDrive/llm_course/day1/quality_note.json", "w", encoding="utf-8") as f:
    json.dump(quality_note, f, ensure_ascii=False, indent=2)

print("✅ 저장 완료 → Day 2에서 이 파일을 사용합니다.")

**📝 품질 불일치 기록표 — 직접 채우세요 (Day 2 입력 데이터)**

| 항목 | 1회차 | 2회차 | 3회차 |
|------|-------|-------|-------|
| 법조문 인용 여부 | | | |
| 답변 구조 (목록형/서술형) | | | |
| 답변 길이 (글자 수) | | | |
| 정확도 (상/중/하) | | | |
| 핵심 차이점 | | | |

---
## STEP 5. 히스토리 관리 주의사항

멀티턴을 쓸 때 주의할 두 가지 문제입니다.

### ① 비용 누적 — 턴이 늘수록 입력 토큰이 쌓입니다

| 턴 수 | 입력 토큰 (예시) |
|-------|----------------|
| 1턴 | ~50 토큰 |
| 3턴 | ~200 토큰 (4배) |
| 10턴 | ~900 토큰 (18배) |

### ② 컨텍스트 윈도우 한계

| 모델 | 최대 길이 | 한국어 기준 |
|------|----------|------------|
| gpt-4o-mini | 128,000 토큰 | 약 4만 글자 |
| EXAONE 3.5 (Day 4) | 32,768 토큰 | 약 1만 글자 |

In [ ]:
# 새 대화를 시작할 때 히스토리 초기화
def reset_conversation():
    global conversation_history
    conversation_history = [
        {"role": "system", "content": "당신은 법률 전문 AI입니다. 쉽게 설명하세요."}
    ]
    print("🔄 대화가 초기화되었습니다.")

reset_conversation()   # 초기화 확인

In [ ]:
# 전체 실습 누적 비용 확인
print("💰 오늘 실습 전체 비용")
tracker.report()

---
## 🔜 다음 시간 예고

> **지금까지 만든 것:** 멀티턴 구조 + 안전 호출 + 품질 불일치 관찰  
> **다음 시간:** 이것들을 합쳐서 **법률 챗봇 v0.1 완성**  
> **Day 2:** 오늘 저장한 품질 불일치 사례로 **프롬프트 기법 실험**
>
> "프롬프트만 바꾸면 품질 불일치가 해결될까?  
> 안 된다면? → Day 4 파인튜닝이 그 답입니다."